In [3]:
from pathlib import Path
import pandas as pd

# ---- paths ----
RAW_INPUT = Path("/Users/jschmidt3/NMDpredictionmodel/Model/TrunCat/data/TOPMed_stopgain.csv")
CV_PATH = Path("/Users/jschmidt3/NMDpredictionmodel/Model/TrunKitten/results/cv_predictions.csv")
OUT_PATH = Path("/Users/jschmidt3/NMDpredictionmodel/Model/TrunKitten/results/cv_predictions_with_ids.csv")

TARGET = "NMD.ESCAPEE"

print("RAW_INPUT:", RAW_INPUT)
print("CV_PATH:", CV_PATH)
print("OUT_PATH:", OUT_PATH)

# ---- load original raw input ----
df = pd.read_csv(RAW_INPUT, low_memory=False)

# ---- recreate valid-target mask ----
y_full = df[TARGET].map({
    "TRUE": 1,
    "FALSE": 0,
    True: 1,
    False: 0,
    "True": 1,
    "False": 0,
    1: 1,
    0: 0,
}).astype("float")

mask = y_full.notna()

df_valid = df.loc[mask].reset_index(drop=True).copy()
df_valid["cv_index"] = range(len(df_valid))

# ---- load CV predictions ----
cv = pd.read_csv(CV_PATH)
cv = cv.rename(columns={"index": "cv_index"})

# ---- recover useful ID columns ----
candidate_id_cols = [
    "cv_index",
    "variantID", "variant_id", "key",
    "contig", "position", "CHROM", "POS",
    "refAllele", "altAllele", "REF_ALLELE", "ALT_ALLELE", "REF", "ALT",
    "txnames", "GENE_ID", "Gene_ID", "gene", "hgnc_symbol",
]

id_cols = [c for c in candidate_id_cols if c in df_valid.columns]
print("ID columns found:", id_cols)

# ---- merge IDs onto predictions ----
cv_with_ids = cv.merge(
    df_valid[id_cols],
    on="cv_index",
    how="left",
    validate="one_to_one",
)

# ---- sanity checks ----
print("Raw input:", df.shape)
print("Valid target rows:", df_valid.shape)
print("CV predictions:", cv.shape)
print("CV + IDs:", cv_with_ids.shape)

label_match = (
    cv_with_ids["y_true"].astype(int).values
    == y_full.loc[mask].astype(int).values
).mean()

print(f"Label match between cv y_true and reconstructed target: {label_match:.6f}")
print(cv_with_ids.head())

# ---- save ----
cv_with_ids.to_csv(OUT_PATH, index=False)
print("Wrote:", OUT_PATH)

RAW_INPUT: /Users/jschmidt3/NMDpredictionmodel/Model/TrunCat/data/TOPMed_stopgain.csv
CV_PATH: /Users/jschmidt3/NMDpredictionmodel/Model/TrunKitten/results/cv_predictions.csv
OUT_PATH: /Users/jschmidt3/NMDpredictionmodel/Model/TrunKitten/results/cv_predictions_with_ids.csv
ID columns found: ['cv_index', 'variantID', 'key', 'contig', 'position', 'CHROM', 'POS', 'refAllele', 'altAllele', 'REF_ALLELE', 'ALT_ALLELE', 'txnames', 'GENE_ID', 'gene', 'hgnc_symbol']
Raw input: (6019, 1034)
Valid target rows: (5749, 1035)
CV predictions: (5749, 4)
CV + IDs: (5749, 18)
Label match between cv y_true and reconstructed target: 1.000000
   cv_index  y_true  oof_prob  oof_pred_at_youden           variantID  \
0         0       1  0.179624                   0  chr1_103537477_C_T   
1         1       0  0.346998                   0  chr1_103547025_C_T   
2         2       1  0.736696                   1   chr1_10463096_G_A   
3         3       0  0.202103                   0  chr1_107766472_G_A   
4    